# Last Model (Copied from Final Model Cell)
Uses the same dataset path and the same code as the last model cell.

In [1]:
# Retrain with more epochs + early stopping
# Frame-based LSTM anomaly detection (5 sensors per timestamp)
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    )

# -------------------------
# Config
# -------------------------
CSV_PATH = "smart_water_minimal_physics_with_sensor_id.csv"
FEATURE_COLS = ["pressure", "flow_rate", "temperature", "pump_power", "pressure_mean", "pressure_var"]
LABEL_COL = "anomaly_detected"
EXPECTED_SENSORS = [1, 2, 3, 4, 5]
SEQ_LEN = 5
BATCH_SIZE = 64
EPOCHS = 12
LR = 1e-3
THRESH = 0.5
RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# -------------------------
# 1) Load & preprocess
# -------------------------
df = pd.read_csv(CSV_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"]).copy()
df = df.sort_values(["timestamp", "sensor_id"]).reset_index(drop=True)

# Ensure required columns exist
required_cols = ["timestamp", "sensor_id", *FEATURE_COLS, LABEL_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Numeric conversion + fill
for col in FEATURE_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df.groupby("sensor_id")[col].transform(lambda s: s.ffill().bfill())
    df[col] = df[col].fillna(df[col].median())

df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce").fillna(0).astype(int)
df[LABEL_COL] = (df[LABEL_COL] > 0).astype(int)

# Verify exactly 5 sensors per timestamp and ids are 1..5
counts = df.groupby("timestamp")["sensor_id"].nunique()
bad_counts = counts[counts != len(EXPECTED_SENSORS)]
if len(bad_counts) > 0:
    raise ValueError(
        f"Found {len(bad_counts)} timestamps without exactly {len(EXPECTED_SENSORS)} sensors. Example: {bad_counts.head().to_dict()}"
    )

sensor_sets_ok = df.groupby("timestamp")["sensor_id"].apply(
    lambda s: set(s.tolist()) == set(EXPECTED_SENSORS)
    )
if not sensor_sets_ok.all():
    bad_ts = sensor_sets_ok[~sensor_sets_ok].index[:5].tolist()
    raise ValueError(f"Some timestamps do not contain sensor ids {EXPECTED_SENSORS}. Examples: {bad_ts}")

# -------------------------
# 2) Build frames: (N_times, 5, n_features)
# -------------------------
grp = df.groupby("timestamp", sort=True)
n_times = grp.ngroups
n_features = len(FEATURE_COLS)

X_frames = np.zeros((n_times, len(EXPECTED_SENSORS), n_features), dtype=np.float32)
y_frames = np.zeros(n_times, dtype=np.int64)
frame_times = []

for i, (ts, g) in enumerate(grp):
    g = g.sort_values("sensor_id")
    X_frames[i] = g[FEATURE_COLS].values.astype(np.float32)
    y_frames[i] = int(g[LABEL_COL].max())  # frame anomaly if any of 5 sensors anomalous
    frame_times.append(ts)

frame_times = np.array(frame_times)
print("X_frames shape:", X_frames.shape)
print("y_frames shape:", y_frames.shape)
print("Frame anomaly rate:", y_frames.mean())

# Standardize by fitting on training-time frames only
split_idx = int(0.8 * n_times)
scaler = StandardScaler()
X_train_flat = X_frames[:split_idx].reshape(-1, n_features)
scaler.fit(X_train_flat)

X_frames_scaled = scaler.transform(X_frames.reshape(-1, n_features)).reshape(X_frames.shape).astype(np.float32)

# -------------------------
# 3) Build LSTM sequences
# -------------------------
def build_sequences(Xf, yf, times, seq_len=5):
    X_seq, y_seq, t_seq = [], [], []
    for i in range(0, len(Xf) - seq_len + 1):
        window = Xf[i:i + seq_len]                        # (seq_len, 5, n_features)
        label = yf[i + seq_len - 1]                      # last frame label
        t_last = times[i + seq_len - 1]

        window_flat = window.reshape(seq_len, -1)        # (seq_len, 5*n_features)
        X_seq.append(window_flat)
        y_seq.append(label)
        t_seq.append(t_last)

    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32), np.array(t_seq)

X_lstm, y_lstm, seq_times = build_sequences(X_frames_scaled, y_frames, frame_times, seq_len=SEQ_LEN)
input_dim = X_lstm.shape[-1]
print("X_lstm shape:", X_lstm.shape, "input_dim:", input_dim)
print("y_lstm shape:", y_lstm.shape)

# Time-respecting split on sequences (earliest 80% train, latest 20% val)
n_seq = len(X_lstm)
split_seq = int(0.8 * n_seq)

X_train, y_train = X_lstm[:split_seq], y_lstm[:split_seq]
X_val, y_val = X_lstm[split_seq:], y_lstm[split_seq:]
t_val = seq_times[split_seq:]

print("Train sequences:", X_train.shape, "Val sequences:", X_val.shape)

# -------------------------
# 4) PyTorch dataset/model
# -------------------------
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMAnomalyClassifier(nn.Module):
    def __init__(self, input_dim, hidden_size=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_h = out[:, -1, :]
        logits = self.fc(self.dropout(last_h)).squeeze(1)
        return logits

def predict_probabilities(model, loader):
    model.eval()
    all_probs = []
    all_true = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_true.append(yb.numpy())
    return np.concatenate(all_true), np.concatenate(all_probs)

def evaluate_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "cm": confusion_matrix(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        metrics["roc_auc"] = float("nan")
    return metrics

train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(SeqDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

import copy

EPOCHS = 10
PATIENCE = 8
MIN_DELTA = 1e-4

train_pos = float(np.sum(y_train))
train_neg = float(len(y_train) - train_pos)
pos_weight_value = train_neg / max(train_pos, 1.0)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

model_es = LSTMAnomalyClassifier(input_dim=input_dim, hidden_size=64, num_layers=1, dropout=0.2).to(DEVICE)
optimizer_es = torch.optim.Adam(model_es.parameters(), lr=LR)
criterion_es = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_loss = np.inf
best_state = None
best_epoch = 0
wait = 0

for epoch in range(1, EPOCHS + 1):
    model_es.train()
    train_running_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_es.zero_grad()
        logits = model_es(xb)
        loss = criterion_es(logits, yb)
        loss.backward()
        optimizer_es.step()
        train_running_loss += loss.item() * len(xb)

    train_loss = train_running_loss / len(train_loader.dataset)

    model_es.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model_es(xb)
            vloss = criterion_es(logits, yb)
            val_running_loss += vloss.item() * len(xb)
    val_loss = val_running_loss / len(val_loader.dataset)

    y_val_true, p_val = predict_probabilities(model_es, val_loader)
    met = evaluate_probs(y_val_true.astype(int), p_val, THRESH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_acc={met['accuracy']:.4f} | val_prec={met['precision']:.4f} | "
        f"val_rec={met['recall']:.4f} | val_f1={met['f1']:.4f}"
    )

    if val_loss < (best_val_loss - MIN_DELTA):
        best_val_loss = val_loss
        best_state = copy.deepcopy(model_es.state_dict())
        best_epoch = epoch
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch} (val_loss={best_val_loss:.4f})")
            break

if best_state is not None:
    model_es.load_state_dict(best_state)

y_val_true, p_val = predict_probabilities(model_es, val_loader)
y_val_pred = (p_val >= THRESH).astype(int)
final_metrics_es = evaluate_probs(y_val_true.astype(int), p_val, THRESH)

print("\n=== Final Validation Metrics (Early Stopping Model) ===")
print(f"Best epoch: {best_epoch} | Best val_loss: {best_val_loss:.4f}")
print("Confusion matrix:\n", final_metrics_es["cm"])
print(
    {k: round(v, 4) if isinstance(v, float) else v
     for k, v in final_metrics_es.items() if k != "cm"}
    )


X_frames shape: (17280, 5, 6)
y_frames shape: (17280,)
Frame anomaly rate: 0.1501736111111111
X_lstm shape: (17276, 5, 30) input_dim: 30
y_lstm shape: (17276,)
Train sequences: (13820, 5, 30) Val sequences: (3456, 5, 30)
Epoch 01/10 | train_loss=0.9461 | val_loss=0.9143 | val_acc=0.9123 | val_prec=0.8182 | val_rec=0.5465 | val_f1=0.6553
Epoch 02/10 | train_loss=0.6750 | val_loss=0.7257 | val_acc=0.9407 | val_prec=0.9066 | val_rec=0.6812 | val_f1=0.7779
Epoch 03/10 | train_loss=0.5582 | val_loss=0.5735 | val_acc=0.9580 | val_prec=0.9548 | val_rec=0.7609 | val_f1=0.8469
Epoch 04/10 | train_loss=0.4821 | val_loss=0.5072 | val_acc=0.9644 | val_prec=0.9698 | val_rec=0.7913 | val_f1=0.8715
Epoch 05/10 | train_loss=0.4336 | val_loss=0.4646 | val_acc=0.9583 | val_prec=0.8869 | val_rec=0.8330 | val_f1=0.8591
Epoch 06/10 | train_loss=0.3958 | val_loss=0.4290 | val_acc=0.9670 | val_prec=0.9275 | val_rec=0.8501 | val_f1=0.8871
Epoch 07/10 | train_loss=0.3764 | val_loss=0.4206 | val_acc=0.9462 | va

In [3]:
import joblib
joblib.dump(scaler, "feature_scaler.pkl")

['feature_scaler.pkl']